# Lesson 6 — Persistent memory with a checkpointer

**Goal:** make the graph remember a conversation across separate `.invoke()`
calls — and across process restarts — without you hand-writing any save/load
code. This is LangGraph's principled answer to your proxy's
`save_turn_background` + `get_recent_messages`.

## The idea: a checkpointer + a thread_id

In Lesson 4 you passed the whole message list into every `.invoke()` and used
a fake list for history. A **checkpointer** flips that around: LangGraph saves
the state after every step, keyed by a `thread_id` you choose (think: one
thread = one user's conversation). Next invoke with the same `thread_id`, it
loads the saved state first. You only pass the *new* message.

### The proxy parallel — `api/proxy/database.py` + `main.py:231-327`
Your proxy does this by hand: `user_id` is effectively the thread key,
`get_recent_messages` is the load, `save_turn_background` is the save, and the
`messages` table is the store. A `SqliteSaver` replaces all three — same
SQLite idea, but the framework owns the read/write.

In [ ]:
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

## 1. A tiny graph (just a model call)
We keep the graph minimal so the focus stays on memory, not nodes.

In [ ]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

model = ChatOpenAI(base_url='http://127.0.0.1:8080/v1', api_key='not-needed',
                   model='qwen3-14b-q4_k_m', temperature=0.5)

def call_model(state: State) -> dict:
    return {'messages': [model.invoke(state['messages'])]}

builder = StateGraph(State)
builder.add_node('call_model', call_model)
builder.add_edge(START, 'call_model')
builder.add_edge('call_model', END)

## 2. Compile WITH a checkpointer

Start with `InMemorySaver` — it keeps state in a Python dict, so memory lasts
as long as the kernel is alive (great for understanding the mechanism). We'll
swap in the SQLite one below.

In [ ]:
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

## 3. Talk to it using a thread_id

The `config` dict carries the `thread_id`. Notice we pass **only the new
message** each time — the checkpointer supplies the prior turns. This is the
payoff: no manual history assembly like your proxy does.

In [ ]:
config = {'configurable': {'thread_id': 'kunal-demo'}}

graph.invoke({'messages': [HumanMessage(content='My name is Kunal and I love Rust.')]}, config)
out = graph.invoke({'messages': [HumanMessage(content="What's my name and what do I love?")]}, config)
print(out['messages'][-1].content)

### Switch threads = switch conversations
A different `thread_id` has its own separate memory — it won't know your name.
This is exactly how `user_id` isolates users in your proxy.

In [ ]:
other = {'configurable': {'thread_id': 'someone-else'}}
out = graph.invoke({'messages': [HumanMessage(content="What's my name?")]}, other)
print(out['messages'][-1].content)

### Peek at the saved state
`get_state` shows what the checkpointer is holding for a thread — the full
message history it will reload next time.

In [ ]:
snap = graph.get_state(config)
for m in snap.values['messages']:
    print(f'[{m.type}] {m.content[:60]}')

## 4. Make it survive restarts — SqliteSaver

`InMemorySaver` forgets everything when the kernel dies. `SqliteSaver` writes
to a `.db` file instead — so a conversation persists across restarts, just
like your `stackmind.db`. The API is identical; only the checkpointer changes.

> Requires the `langgraph-checkpoint-sqlite` package. If it's not installed:
> `.venv\Scripts\python -m pip install langgraph-checkpoint-sqlite`
> (it may already have come in with langgraph; the cell handles both).

In [ ]:
try:
    from langgraph.checkpoint.sqlite import SqliteSaver
    import sqlite3
    conn = sqlite3.connect('langgraph_memory.db', check_same_thread=False)
    sqlite_saver = SqliteSaver(conn)
    durable_graph = builder.compile(checkpointer=sqlite_saver)

    cfg = {'configurable': {'thread_id': 'persistent-kunal'}}
    durable_graph.invoke({'messages': [HumanMessage(content='Remember the number 42.')]}, cfg)
    out = durable_graph.invoke({'messages': [HumanMessage(content='What number did I ask you to remember?')]}, cfg)
    print(out['messages'][-1].content)
    print('\nSaved to langgraph_memory.db — re-run this cell after restarting the kernel and it still remembers.')
except ImportError:
    print('SqliteSaver not installed. Run:')
    print('  .venv\\Scripts\\python -m pip install langgraph-checkpoint-sqlite')

---
**The mapping to your proxy, made concrete:**

| Your proxy | LangGraph |
|---|---|
| `user_id` | `thread_id` |
| `get_recent_messages` | checkpointer auto-loads |
| `save_turn_background` | checkpointer auto-saves |
| `messages` table in `stackmind.db` | `SqliteSaver`'s tables |

Your proxy also compresses old turns into a rolling summary
(`_generate_summary`). LangGraph doesn't do that for free — but now you can
see it'd be *another node* in the graph that rewrites state, which is a clean
place to put it. That's a good thing to try building yourself next.